<a href="https://colab.research.google.com/github/chandrakalagunda/Chandrakala/blob/main/Traffic_demand_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder
!pip install catboost
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.8 MB/s eta 0:00:00


In [ ]:
# Load data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

In [ ]:
target = "demand"

In [ ]:
# Combine for same preprocessing
test[target] = np.nan
data = pd.concat([train, test], axis=0, ignore_index=True)

In [ ]:
# Missing value handling
cat_cols = ["geohash", "RoadType", "LargeVehicles", "Landmarks", "Weather"]

for col in cat_cols:
    data[col] = data[col].fillna("Unknown").astype(str)

data["Temperature"] = data["Temperature"].fillna(data["Temperature"].median())

In [ ]:
# Time feature engineering
data[["hour", "minute"]] = data["timestamp"].str.split(":", expand=True).astype(int)

data["time_minutes"] = data["hour"] * 60 + data["minute"]

data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)

data["time_sin"] = np.sin(2 * np.pi * data["time_minutes"] / 1440)
data["time_cos"] = np.cos(2 * np.pi * data["time_minutes"] / 1440)

data["is_morning_peak"] = data["hour"].isin([7, 8, 9, 10]).astype(int)
data["is_evening_peak"] = data["hour"].isin([17, 18, 19, 20]).astype(int)
data["is_night"] = ((data["hour"] >= 0) & (data["hour"] <= 5)).astype(int)

In [ ]:
# Interaction features
data["geo_hour"] = data["geohash"] + "_" + data["hour"].astype(str)
data["geo_time"] = data["geohash"] + "_" + data["timestamp"]
data["road_weather"] = data["RoadType"] + "_" + data["Weather"]
data["geo_weather"] = data["geohash"] + "_" + data["Weather"]

data["bad_weather"] = data["Weather"].isin(["Rainy", "Foggy", "Snowy"]).astype(int)
data["large_vehicle_allowed"] = (data["LargeVehicles"] == "Allowed").astype(int)
data["has_landmark"] = (data["Landmarks"] == "Yes").astype(int)

data["lane_weather"] = data["NumberofLanes"] * (data["bad_weather"] + 1)
data["capacity_score"] = data["NumberofLanes"] * (data["large_vehicle_allowed"] + 1)

In [ ]:
# Frequency encoding
for col in ["geohash", "geo_hour", "geo_time", "RoadType", "Weather"]:
    freq = data[col].value_counts()
    data[col + "_freq"] = data[col].map(freq)

In [ ]:
# Target encoding using train only
train_part = data[data[target].notna()].copy()

for col in ["geohash", "timestamp", "hour", "geo_hour", "geo_time", "RoadType", "Weather"]:
    mean_map = train_part.groupby(col)[target].mean()
    data[col + "_mean_demand"] = data[col].map(mean_map)

# Fill missing target encoding values
for col in data.columns:
    if col.endswith("_mean_demand"):
        data[col] = data[col].fillna(train[target].mean())

In [ ]:
# Previous day same location-time demand
# Powerful feature for day 49 prediction
day48 = train[train["day"] == 48][["geohash", "timestamp", "demand"]]
prev_map = day48.set_index(["geohash", "timestamp"])["demand"]

data["prev_day_same_time_demand"] = data.set_index(["geohash", "timestamp"]).index.map(prev_map)
data["prev_day_same_time_demand"] = data["prev_day_same_time_demand"].fillna(train[target].mean())


In [ ]:
# Encode categorical columns for XGBoost
# CatBoost will use original categorical values
data_xgb = data.copy()

encode_cols = [
    "geohash", "timestamp", "RoadType", "LargeVehicles", "Landmarks",
    "Weather", "geo_hour", "geo_time", "road_weather", "geo_weather"
]

for col in encode_cols:
    le = LabelEncoder()
    data_xgb[col] = le.fit_transform(data_xgb[col].astype(str))

In [ ]:
# Split back
train_cb = data[data[target].notna()].copy()
test_cb = data[data[target].isna()].copy()

train_xgb = data_xgb[data_xgb[target].notna()].copy()
test_xgb = data_xgb[data_xgb[target].isna()].copy()

drop_cols = ["Index", target]

X_cb = train_cb.drop(columns=drop_cols)
y = train_cb[target]
X_test_cb = test_cb.drop(columns=drop_cols)

X_xgb = train_xgb.drop(columns=drop_cols)
X_test_xgb = test_xgb.drop(columns=drop_cols)


In [ ]:
# Validation split
X_train_cb, X_val_cb, y_train, y_val = train_test_split(
    X_cb, y, test_size=0.15, random_state=42
)

X_train_xgb, X_val_xgb, _, _ = train_test_split(
    X_xgb, y, test_size=0.15, random_state=42
)

cat_features = [
    "geohash", "timestamp", "RoadType", "LargeVehicles", "Landmarks",
    "Weather", "geo_hour", "geo_time", "road_weather", "geo_weather"
]

cat_feature_indices = [X_cb.columns.get_loc(col) for col in cat_features]

In [ ]:
# CatBoost Model
cat_model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    loss_function="RMSE",
    eval_metric="R2",
    random_seed=42,
    verbose=200,
    early_stopping_rounds=200
)

cat_model.fit(
    X_train_cb,
    y_train,
    eval_set=(X_val_cb, y_val),
    cat_features=cat_feature_indices
)

cat_val_pred = cat_model.predict(X_val_cb)
print("CatBoost R2:", r2_score(y_val, cat_val_pred))

0:	learn: 0.0555572	test: 0.0558098	best: 0.0558098 (0)	total: 209ms	remaining: 10m 26s
200:	learn: 0.9969922	test: 0.9968124	best: 0.9968124 (200)	total: 42.6s	remaining: 9m 53s
400:	learn: 0.9984873	test: 0.9981286	best: 0.9981286 (400)	total: 1m 29s	remaining: 9m 38s
600:	learn: 0.9989747	test: 0.9984586	best: 0.9984586 (600)	total: 2m 7s	remaining: 8m 29s
800:	learn: 0.9992013	test: 0.9985719	best: 0.9985719 (800)	total: 2m 47s	remaining: 7m 40s
1000:	learn: 0.9993421	test: 0.9986051	best: 0.9986067 (975)	total: 3m 24s	remaining: 6m 48s
1200:	learn: 0.9994380	test: 0.9986212	best: 0.9986228 (1188)	total: 4m 1s	remaining: 6m 1s
1400:	learn: 0.9995171	test: 0.9986277	best: 0.9986279 (1273)	total: 4m 38s	remaining: 5m 17s
1600:	learn: 0.9995815	test: 0.9986295	best: 0.9986295 (1600)	total: 5m 15s	remaining: 4m 35s
1800:	learn: 0.9996387	test: 0.9986325	best: 0.9986342 (1781)	total: 5m 51s	remaining: 3m 54s
2000:	learn: 0.9996788	test: 0.9986352	best: 0.9986377 (1831)	total: 6m 30s	rem

In [ ]:
# XGBoost Model
xgb_model = XGBRegressor(
    n_estimators=2500,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="reg:squarederror",
    random_state=42,
    tree_method="hist"
)

xgb_model.fit(
    X_train_xgb,
    y_train,
    eval_set=[(X_val_xgb, y_val)],
    verbose=200
)

xgb_val_pred = xgb_model.predict(X_val_xgb)
print("XGBoost R2:", r2_score(y_val, xgb_val_pred))

[0]	validation_0-rmse:0.13810
[200]	validation_0-rmse:0.00560
[400]	validation_0-rmse:0.00556
[600]	validation_0-rmse:0.00556
[800]	validation_0-rmse:0.00556
[1000]	validation_0-rmse:0.00556
[1200]	validation_0-rmse:0.00555
[1400]	validation_0-rmse:0.00555
[1600]	validation_0-rmse:0.00555
[1800]	validation_0-rmse:0.00555
[2000]	validation_0-rmse:0.00555
[2200]	validation_0-rmse:0.00555
[2400]	validation_0-rmse:0.00555
[2499]	validation_0-rmse:0.00555
XGBoost R2: 0.9984774905638305


In [ ]:
# Ensemble Validation
ensemble_val = 0.7 * cat_val_pred + 0.3 * xgb_val_pred
print("Ensemble R2:", r2_score(y_val, ensemble_val))

Ensemble R2: 0.9989172572922581


In [ ]:
# Train final models on full train data
cat_model.fit(
    X_cb,
    y,
    cat_features=cat_feature_indices,
    verbose=200
)

xgb_model.fit(
    X_xgb,
    y,
    verbose=200
)


0:	learn: 0.0555546	total: 475ms	remaining: 23m 44s
200:	learn: 0.9969775	total: 41.3s	remaining: 9m 35s
400:	learn: 0.9984400	total: 1m 20s	remaining: 8m 42s
600:	learn: 0.9988498	total: 1m 58s	remaining: 7m 54s
800:	learn: 0.9990956	total: 2m 38s	remaining: 7m 15s
1000:	learn: 0.9992631	total: 3m 19s	remaining: 6m 38s
1200:	learn: 0.9993789	total: 3m 59s	remaining: 5m 58s
1400:	learn: 0.9994651	total: 4m 40s	remaining: 5m 19s
1600:	learn: 0.9995305	total: 5m 19s	remaining: 4m 39s
1800:	learn: 0.9995895	total: 5m 58s	remaining: 3m 58s
2000:	learn: 0.9996314	total: 6m 38s	remaining: 3m 18s
2200:	learn: 0.9996751	total: 7m 17s	remaining: 2m 38s
2400:	learn: 0.9997126	total: 7m 58s	remaining: 1m 59s
2600:	learn: 0.9997399	total: 8m 37s	remaining: 1m 19s
2800:	learn: 0.9997663	total: 9m 17s	remaining: 39.6s
2999:	learn: 0.9997848	total: 9m 57s	remaining: 0us


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.85, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=2500,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
# Predict test
cat_pred = cat_model.predict(X_test_cb)
xgb_pred = xgb_model.predict(X_test_xgb)

final_pred = 0.7 * cat_pred + 0.3 * xgb_pred
final_pred = np.clip(final_pred, 0, None)

In [ ]:
# Submission
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": final_pred
})

submission.to_csv("submission.csv", index=False)

In [ ]:
print("submission.csv created successfully!")
print(submission.head())

submission.csv created successfully!
   Index    demand
0      0  0.095991
1      1  0.090228
2      2  0.092483
3      3  0.072151
4      4  0.106363


In [ ]:
print(submission.shape)

(41778, 2)


In [ ]:
print(submission.columns)

Index(['Index', 'demand'], dtype='object')
